# 05 — Tourism

Visitor arrivals per resident, for every Pacific island the Pacific Data Hub
covers.

The ratio is the point. Absolute arrival counts put Fiji and Guam at the top
and say nothing a reader can feel; dividing by the resident population says how
many outsiders each island hosts per person — ten in the Cook Islands, one in
Fiji, one in sixty in Papua New Guinea. It is the same move the ASR makes on
emissions, on a quantity the islands *receive* rather than cause.

SPC publishes arrivals in **two** dataflows, and they are not the same measure:

| Dataflow | What it counts | Years | Role here |
|---|---|---|---|
| `DF_TOURISM_ARRIVALS` | overnight tourists (`TOUR`) and same-day visitors (`EXCR`), separately | 2000–2023 | the long series, primary everywhere it has a figure |
| `DF_OVERSEAS_VISITORS` | every overseas arrival, air and sea, cruise passengers included — total (`_T`) 2005–2023, split into tourists (`TOU`) / excursionists (`EXC`) only from 2018 | 2005–2023 | its `TOU` split fills 13 recent gaps the long series leaves |

The headline measure is **overnight tourists**: `TOUR` from the first dataflow,
`TOU` from the second. The `_T` total is carried alongside it as
`all_visitors_per_capita` rather than replacing it — it covers more island-years
but folds cruise-ship excursionists in, which roughly doubles Vanuatu and
triples New Caledonia, and it is not comparable to the world series in
section 4. Two columns, two questions: how many people stay, how many arrive.

(SPC's third tourism dataflow, `DF_TOURISM_EARNINGS`, is money — gross earnings,
per visitor and as a share of GDP — not arrivals.)

Population is SPC's own mid-year estimate (`DF_POP_PROJ`), not the World Bank
table the rest of the pipeline uses — section 2 explains why, and checks the
two agree.

**Outputs**
- `data_viz/tourism.csv` — the island-year panel
- `data_viz/tourism.json` — 2019 snapshot, latest year, and the world ranking

In [ ]:
import json

import pandas as pd
import pycountry
import requests

from config import VIZ, WB_POP, YEARS
from pdh_api import fetch_data_pacific, fetch_structure_pacific

START, END = str(min(YEARS)), str(max(YEARS))

# GEO_PICT mixes regional aggregates in with the islands. Pitcairn (PN) stays
# in — it is a real place with 50 residents — but has no tourism figures and
# drops out on the merge in section 3.
AGGREGATES = {"_T", "_TXPNG", "MEL", "MELXPNG", "MIC", "POL"}

## 1. Arrivals — Pacific Data Hub

`DF_TOURISM_ARRIVALS` splits arrivals into **overnight tourists** (`TOUR`) and
**same-day visitors** (`EXCR`) — mostly cruise passengers ashore for the day.
This notebook counts overnight tourists only. Excursionists are the patchier
series (13 islands, and nothing at all for the Marshalls, Solomons or Niue),
they are not comparable to the World Bank's world series in section 4, and a
cruise call ashore is not the same visit as a two-week stay. `EXCR` is one
`key` change away if the story ever wants it — in Vanuatu and New Caledonia it
is the larger number.

In [ ]:
arrivals = fetch_data_pacific(
    source="DF_TOURISM_ARRIVALS",
    start_period=START,
    end_period=END,
    key="A..TOUR",
)

arrivals = (
    arrivals[~arrivals["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"GEO_PICT": "pict", "value": "tourists"})
    [["pict", "year", "tourists"]]
    .dropna(subset=["tourists"])
)

print(f"{arrivals['pict'].nunique()} islands, {len(arrivals)} island-years, "
      f"{arrivals['year'].min()}-{arrivals['year'].max()}")
print(arrivals.groupby("pict")["year"].agg(["min", "max", "count"]).to_string())

The long series thins out after 2021 — most islands stop there, and the 2022–23
recovery is missing for all but Fiji, New Caledonia and Guam. The tourist split
of `DF_OVERSEAS_VISITORS` (v2.0) runs 2018 onwards and fills those gaps on the
same measure.

Where both report the same island-year they mostly agree: median disagreement
0.2%. Twelve of 67 overlapping rows differ by more than 5% — French Polynesia
runs ~15–19% apart across 2019–21, a definitional gap rather than a revision —
and one row is plainly corrupt: `DF_OVERSEAS_VISITORS` puts the Marshall
Islands at 6.76e14 arrivals in 2018, seventeen billion per resident, against
6,800 in the long series.

So the long series wins wherever it has a figure, and the recent one is used
only to fill holes. Every filled row is checked against the island's population
in section 3.

In [ ]:
recent = fetch_data_pacific(
    source="DF_OVERSEAS_VISITORS",
    start_period=START,
    end_period=END,
    key="A..TOU.NOSVA",
    v="2.0",
)

recent = (
    recent[~recent["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"GEO_PICT": "pict", "value": "tourists_recent"})
    [["pict", "year", "tourists_recent"]]
    .dropna(subset=["tourists_recent"])
)

overlap = arrivals.merge(recent, on=["pict", "year"])
gap = (overlap["tourists_recent"] - overlap["tourists"]).abs() / overlap["tourists"]
print(f"{len(overlap)} overlapping island-years, median disagreement "
      f"{gap.median():.2%}, {(gap > 0.05).sum()} rows over 5%")

In [ ]:
tourists = arrivals.merge(recent, on=["pict", "year"], how="outer")
tourists["source"] = tourists["tourists"].notna().map(
    {True: "DF_TOURISM_ARRIVALS", False: "DF_OVERSEAS_VISITORS"}
)
tourists["tourists"] = tourists["tourists"].fillna(tourists["tourists_recent"])
tourists = tourists.drop(columns="tourists_recent")

print(tourists["source"].value_counts().to_string())
print(tourists[tourists["source"] == "DF_OVERSEAS_VISITORS"]
      .sort_values(["pict", "year"]).to_string(index=False))

### The second measure: every arrival

`DF_OVERSEAS_VISITORS` also carries a total (`_T`) — air and sea, cruise
excursionists included — running 2005–2023 and covering fourteen islands in
2023 against eight for overnight tourists.

It is kept as a **separate column, not a replacement**. The two measures are
identical for the islands at the top of the ranking (Cook Islands, Guam, Niue,
French Polynesia move less than 1%) and diverge exactly where cruise ships call:
New Caledonia 0.46 to 1.67 visitors per resident in 2019, American Samoa 0.38 to
1.17, Vanuatu 0.41 to 0.88. Overnight tourists stay the headline because they
are what the World Bank world series in section 4 counts; the total is there for
any scene that wants the cruise gap.

In [ ]:
totals = fetch_data_pacific(
    source="DF_OVERSEAS_VISITORS",
    start_period=START,
    end_period=END,
    key="A.._T.NOSVA",  # _T = tourists + excursionists, air and sea
    v="2.0",
)

totals = (
    totals[~totals["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"GEO_PICT": "pict", "value": "all_visitors"})
    [["pict", "year", "all_visitors"]]
    .dropna(subset=["all_visitors"])
)

tourists = tourists.merge(totals, on=["pict", "year"], how="outer")
print(f"{totals['pict'].nunique()} islands, {totals['year'].min()}-{totals['year'].max()}, "
      f"{tourists['all_visitors'].notna().sum()} island-years with a total")

## 2. Population — SPC mid-year estimates

The rest of the pipeline divides by the World Bank population table, because
pyaesa allocates carbon budgets off it. That table cannot be used here: it has
no entry for American Samoa, Guam, the Northern Mariana Islands, the Cook
Islands, Niue, Tokelau or Wallis and Futuna — and those are exactly the islands
with the highest tourism ratios. Dropping them would delete the finding.

`DF_POP_PROJ` is SPC's own mid-year estimate and covers all of them, every year
from 2000. For the 14 islands both tables carry, they are the same numbers to
within 0.02% at the median — so mixing this denominator with the World Bank one
used elsewhere does not create a discontinuity between notebooks.

In [ ]:
pop = fetch_data_pacific(
    source="DF_POP_PROJ",
    start_period=START,
    end_period=END,
    key="A..MIDYEARPOPEST._T._T",  # mid-year estimate, both sexes, all ages
    v="3.0",
)

pop = (
    pop[~pop["GEO_PICT"].isin(AGGREGATES)]
    .assign(year=lambda d: d["TIME_PERIOD"].astype(int))
    .rename(columns={"GEO_PICT": "pict", "value": "population"})
    [["pict", "year", "population"]]
)

# SPC's own labels for the islands, and ISO3 so the app can join this to
# countries.csv and the ASR tables.
names = fetch_structure_pacific("DF_POP_PROJ")["dimensions"]["GEO_PICT"]
iso3 = {c: pycountry.countries.get(alpha_2=c).alpha_3 for c in pop["pict"].unique()}

print(f"{pop['pict'].nunique()} islands, {len(pop)} island-years")

In [ ]:
wb = (
    pd.read_csv(WB_POP)
    .query("variable == 'Population'")
    .melt(id_vars="iso3_code", value_vars=[str(y) for y in YEARS],
          var_name="year", value_name="wb_population")
    .astype({"year": int})
    .rename(columns={"iso3_code": "iso_code"})
)

check = (
    pop.assign(iso_code=lambda d: d["pict"].map(iso3))
    .merge(wb, on=["iso_code", "year"])
    .assign(diff=lambda d: (d["population"] - d["wb_population"]).abs()
            / d["wb_population"])
)

print(f"{check['iso_code'].nunique()} of {pop['pict'].nunique()} islands are in "
      f"the World Bank table")
print(f"median gap {check['diff'].median():.2%}, max {check['diff'].max():.2%} "
      f"({check.loc[check['diff'].idxmax(), 'iso_code']})")

## 3. Tourists per capita

One division. The merge is inner, so an island needs both an arrival count and
a population estimate to appear — which is what drops Pitcairn and Nauru
(neither reports tourism to SPC).

In [ ]:
tourism = (
    tourists.merge(pop, on=["pict", "year"])
    .assign(
        iso_code=lambda d: d["pict"].map(iso3),
        name=lambda d: d["pict"].map(names),
        tourists_per_capita=lambda d: d["tourists"] / d["population"],
        all_visitors_per_capita=lambda d: d["all_visitors"] / d["population"],
    )
    .sort_values(["pict", "year"])
    [["pict", "iso_code", "name", "year", "tourists", "tourists_per_capita",
      "all_visitors", "all_visitors_per_capita", "population", "source"]]
)

# Guard against another 6.76e14: no island has ever hosted 25 tourists per
# resident, so anything above that is a unit error, not a record year.
implausible = tourism[(tourism["tourists_per_capita"] > 25)
                      | (tourism["all_visitors_per_capita"] > 25)]
if len(implausible):
    print("dropped as implausible:")
    print(implausible.to_string(index=False))
    tourism = tourism.drop(implausible.index)

# A few island-years have a total but no overnight figure (Nauru, and Fiji
# before 2000's start). They carry no tourists_per_capita and are dropped.
tourism = tourism.dropna(subset=["tourists_per_capita"])

print(f"{tourism['pict'].nunique()} islands, {len(tourism)} island-years, "
      f"{tourism['all_visitors_per_capita'].notna().sum()} with an all-visitor figure")

In [ ]:
YEAR_SNAP = 2019  # the last year before COVID closed the borders

snapshot = (
    tourism[tourism["year"] == YEAR_SNAP]
    .sort_values("tourists_per_capita", ascending=False)
)
print(f"{YEAR_SNAP}")
print(snapshot[["name", "population", "tourists_per_capita",
                "all_visitors_per_capita"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

In [ ]:
latest = (
    tourism.sort_values("year")
    .groupby("pict")
    .tail(1)
    .sort_values("tourists_per_capita", ascending=False)
)
print("most recent year on record, per island")
print(latest[["name", "year", "tourists", "tourists_per_capita"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 4. Where that sits in the world

The World Bank's `ST.INT.ARVL` is the same measure — international tourist
arrivals, overnight — for every country that reports it. It effectively ends in
2019: 220 countries that year, 132 in 2020, nothing usable after. That makes
2019 the only year a Pacific-vs-world ranking can be built on, which is the
other reason the snapshot above is 2019.

Pacific figures come from the Pacific Data Hub even where the World Bank has
the country, so the islands are ranked on SPC's own numbers.

In [ ]:
WB_ARRIVALS = (
    "https://api.worldbank.org/v2/country/all/indicator/ST.INT.ARVL"
    f"?format=json&per_page=20000&date={YEAR_SNAP}"
)

try:
    payload = requests.get(WB_ARRIVALS, timeout=60).json()
except Exception as exc:
    # python.org builds on macOS ship without CA certificates until you run
    # /Applications/Python\ 3.x/Install\ Certificates.command
    print(f"Direct read failed ({type(exc).__name__}); retrying without TLS verification.")
    import urllib3

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    payload = requests.get(WB_ARRIVALS, timeout=60, verify=False).json()

world = pd.DataFrame([
    {"iso_code": row["countryiso3code"], "tourists": row["value"]}
    for row in payload[1]
    if row["value"] is not None and len(row["countryiso3code"]) == 3
])

world = (
    world.merge(wb[wb["year"] == YEAR_SNAP], on="iso_code")
    .assign(tourists_per_capita=lambda d: d["tourists"] / d["wb_population"])
)

print(f"{len(world)} countries from the World Bank for {YEAR_SNAP}")

In [ ]:
ranking = (
    pd.concat([
        world.loc[~world["iso_code"].isin(snapshot["iso_code"]),
                  ["iso_code", "tourists_per_capita"]],
        snapshot[["iso_code", "tourists_per_capita"]],
    ])
    .sort_values("tourists_per_capita", ascending=False)
    .reset_index(drop=True)
)
ranking["rank"] = ranking.index + 1
ranking["n"] = len(ranking)

pacific_rank = ranking[ranking["iso_code"].isin(snapshot["iso_code"])].merge(
    snapshot[["iso_code", "name"]], on="iso_code"
)

print(f"{len(ranking)} countries ranked")
print(pacific_rank[["rank", "name", "tourists_per_capita"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 5. What this explains about Palau — and what it does not

Palau is the reason this notebook exists: it carries the highest ASR in the
project (97× its fair share on the Pacific Data Hub's 82.5 t CO₂-eq per person
in 2023), and hosting five visitors per resident is the obvious candidate
explanation.

The candidate does not survive its own test. Tourism fell 96% between 2019 and
2021 and Palau's reported emissions rose in every one of those years. Across the
whole window the correlation between visitors per resident and emissions per
resident is **0.02** — no relationship at all. Fiji comes out at −0.00, New
Caledonia at −0.36.

So the honest claim is structural, not causal: Palau runs power, water and
transport infrastructure sized for a population several times larger than the
17,772 people the emissions are divided by. Territorial accounting charges all
of it to the residents. That argument stands on how the ratio is built, and it
does not need — and is not supported by — a year-to-year correlation.

In [ ]:
emissions = pd.read_csv(VIZ / "variables.csv")

check = (
    emissions[["iso_code", "year", "emissions_t_per_capita", "asr_eg"]]
    .merge(tourism[["iso_code", "year", "tourists_per_capita"]],
           on=["iso_code", "year"])
)

for iso in ["PLW", "FJI", "NCL"]:
    c = check[check["iso_code"] == iso].dropna()
    r = c["emissions_t_per_capita"].corr(c["tourists_per_capita"])
    print(f"{iso}: corr(emissions per capita, visitors per capita) = {r:+.2f} "
          f"over {len(c)} years")

print()
print(check[(check["iso_code"] == "PLW") & (check["year"] >= 2018)]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 6. Write outputs

In [ ]:
VIZ.mkdir(exist_ok=True)
tourism.to_csv(VIZ / "tourism.csv", index=False)

payload = {
    "meta": {
        "measure": "international overnight tourist arrivals per resident",
        "second_measure": (
            "all_visitors_per_capita adds cruise excursionists and every other "
            "sea arrival; comparable across the Pacific but not to the world "
            "ranking, which counts overnight tourists only"
        ),
        "snapshot_year": YEAR_SNAP,
        "snapshot_note": (
            "2019 is the last pre-COVID year and the last year the World Bank "
            "series covers, so it is both the fair Pacific comparison and the "
            "only possible world ranking"
        ),
        "islands": int(tourism["pict"].nunique()),
        "years": [int(tourism["year"].min()), int(tourism["year"].max())],
        "ranked_countries": int(len(ranking)),
        "sources": (
            "Pacific Data Hub .Stat (SPC) DF_TOURISM_ARRIVALS, "
            "DF_OVERSEAS_VISITORS, DF_POP_PROJ; World Bank ST.INT.ARVL and "
            "population for the world ranking"
        ),
    },
    "snapshot": json.loads(
        snapshot.merge(ranking[["iso_code", "rank"]], on="iso_code", how="left")
        [["iso_code", "name", "tourists", "population", "tourists_per_capita",
          "all_visitors", "all_visitors_per_capita", "rank"]]
        .to_json(orient="records")
    ),
    "latest": json.loads(
        latest[["iso_code", "name", "year", "tourists", "population",
                "tourists_per_capita", "all_visitors_per_capita"]]
        .to_json(orient="records")
    ),
    "series": json.loads(
        tourism[["iso_code", "year", "tourists", "population",
                 "tourists_per_capita", "all_visitors_per_capita"]]
        .to_json(orient="records")
    ),
}

(VIZ / "tourism.json").write_text(json.dumps(payload))

print(f"{len(tourism)} rows -> data_viz/tourism.csv")
print(f"{len(payload['snapshot'])} islands in the {YEAR_SNAP} snapshot -> data_viz/tourism.json")

## 7. Caveats

- **Arrivals, not people.** Every crossing counts, so a resident's returning
  relatives and a consultant's four trips are four tourists. The ratio is
  visits per resident per year, not the share of the population that is a
  visitor on any given day.
- **The window includes COVID.** 2020–21 is a collapse to near zero on every
  island and 2022–23 is a partial recovery reported by only some. Use 2019 for
  any comparison; use the full panel only to show the shock.
- **Latest year varies by island.** Fiji, Samoa, Tonga, Tuvalu, Kiribati, New
  Caledonia, Vanuatu and Guam reach 2023; several others stop at 2021.
- **Two measures, one ranking.** `tourists_per_capita` is overnight visitors
  and is the only column comparable to the world ranking in section 4.
  `all_visitors_per_capita` adds cruise excursionists and every other sea
  arrival, starts in 2005, and is missing for the islands and years SPC's total
  series does not reach. Never mix the two in one chart axis.
- **Territories are in.** Guam, American Samoa, the Northern Mariana Islands,
  the Cook Islands, Niue, Tokelau and Wallis and Futuna appear here but not in
  the ASR notebooks, which are limited to World Bank countries. Joining this
  file to `asr.csv` on `iso_code` will leave them unmatched.
- **Nauru and Pitcairn are absent.** Pitcairn reports no tourism to SPC at all;
  Nauru appears only in the `_T` total of `DF_OVERSEAS_VISITORS`, and only for
  2007–2013 plus 2016, so it has no overnight-tourist series to use.
- **The world ranking is 2019 only.** `ST.INT.ARVL` has no usable coverage
  after 2020, so there is no way to place the islands globally in a recovery
  year.